## Named Entity Recognition Project for NLP
### Nicholas Woody

1) First I will import all necessary libraries. `!pip install` is used to install different modules on Google CoLab. You may need to configure your notebook to be able to access huggingface with a user access token. Instructions can be found [here](https://huggingface.co/docs/hub/security-tokens).

In [13]:
!pip install datasets==3.6.0
!pip install evaluate
!pip install seqeval
from datasets import load_dataset
import nltk
from nltk import pos_tag, ne_chunk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
from transformers import AutoTokenizer, DataCollatorForTokenClassification, AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
import torch

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


2. Then, I will load the dataset and split it into a test and training dataset. I will also review the features of this dataset to familiarize myself with them.

In [14]:

dataset = load_dataset("eriktks/conll2003", trust_remote_code=True)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [15]:
print(train_dataset.features)

print("\n", train_dataset.features['ner_tags'])


{'id': Value(dtype='string', id=None), 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'pos_tags': Sequence(feature=ClassLabel(names=['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'], id=None), length=-1, id=None), 'chunk_tags': Sequence(feature=ClassLabel(names=['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'], id=None), length=-1, id=None), 'ner_tags': Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None), length=-1, id=None)}

 Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-

3) For my first NER technique I will build a dictionary of Named Entities. They key will be the word and the value will be the type of entity. I will loop through the training set, ignoring common punctuation, and add any tokens to the dictionary the first time it encounters them. If the dictionary has encountered a word before it will not add it again, but go with the first type of entity it encountered.

In [16]:

# List of common punctuation
Punctuation = ["'", '"', "!", ".", "(", ")", ":", "[", "]", "&", "*", "$", "@", "^", ",", "?", "-"]

# Empty Dictionary
NER = {}

# Loop through the training dataset to fill the dictionary.
for i in range(len(train_dataset)):
  tokens = train_dataset[i]['tokens']
  labels = train_dataset[i]['ner_tags']
  for j in range(len(labels)):
    if labels[j] != 0:
      if tokens[j] not in Punctuation and tokens[j] not in NER:
        NER[tokens[j]] = labels[j]

3A) With the dictionary created it's time to review the performance.

In [17]:
# Create variables for true positive, false positive, true negative, and false negative.
TP = 0
FP = 0
TN = 0
FN = 0
words = 0 # This is to check my loop is working correctly

# Loop through the test_dataset
for i in range(len(test_dataset)):
  tokens = test_dataset[i]['tokens']
  labels = test_dataset[i]['ner_tags']

  for j in range(len(tokens)):
    words += 1

    # Check to see if token is in dictionary
    # If in dictionary, check to see if the key matches the label
    if tokens[j] in NER:
      if NER[tokens[j]] == labels[j]:
        TP += 1
      elif NER[tokens[j]] != labels[j]:
        FP += 1
    else:
      if labels[j] == 0:
        TN += 1
      elif labels[j] != 0:
        FN += 1


In [18]:
print(f"There are {words} words and {TP + FP + TN + FN} results")

precision = TP / (TP + FP)
recall = TP / (TP + FN)
accuracy = (TP + TN) / (TP + TN + FP + FN)
F1 = (2 * precision * recall) / (precision + recall)

print(f"Precision: {precision:.6f}")
print(f"Recall: {recall:.6f}")
print(f"Accuracy: {accuracy:.6f}")
print(f"F1: {F1:.6f}")


There are 46435 words and 46435 results
Precision: 0.326847
Recall: 0.606897
Accuracy: 0.763993
F1: 0.424875


3B) Now I will review performance based off entity type, independent of B and I tags.

In [19]:
TP = 0
FP = 0
TN = 0
FN = 0
words = 0

for i in range(len(test_dataset)):
  tokens = test_dataset[i]['tokens']
  labels = test_dataset[i]['ner_tags']
  for j in range(len(tokens)):
    words += 1
    if tokens[j] in NER:
      if NER[tokens[j]] in [1, 2] and labels[j] in [1,2]:
        TP += 1
      elif NER[tokens[j]] in [3,4] and labels[j] in [3,4]:
        TP += 1
      elif NER[tokens[j]] in [5,6] and labels[j] in [5,6]:
        TP += 1
      elif NER[tokens[j]] in [7,8] and labels[j] in [7,8]:
        TP += 1
      else:
        FP += 1
    else:
      if labels[j] == 0:
        TN += 1
      elif labels[j] != 0:
        FN += 1

print(f"There are {words} words and {TP + FP + TN + FN} results")

precision = TP / (TP + FP)
recall = TP / (TP + FN)
accuracy = (TP + TN) / (TP + TN + FP + FN)
F1 = (2 * precision * recall) / (precision + recall)

print(f"Precision: {precision:.6f}")
print(f"Recall: {recall:.6f}")
print(f"Accuracy: {accuracy:.6f}")
print(f"F1: {F1:.6f}")

There are 46435 words and 46435 results
Precision: 0.377473
Recall: 0.640674
Accuracy: 0.777495
F1: 0.475053


4) I will use NLTK's built in ne_chunk for NER processing. I reviewed the results of ne_chunk and found that "PERSON", "ORGANIZATION", and "LOCATION" are all valid tags. "GSP" appears to represent location as well. All other tags will be categorized as "MISC".I'll need to process the results to fit the dataset's categories, including the B an I tags. Documentation for this method can be found [here](https://www.nltk.org/api/nltk.chunk.html) and is discussed in chapter 7 of the [nltk book](https://www.nltk.org/book/ch07.html) .

In [20]:
# First I'll create a list of lists (of lists) using NLTKs ne_chunk algorithm
# This is to get the NER tags in a format similar to the test data

nltk_lst = []


# Loop through all test sentences and turn to a named_entity list
# No need to tokenize, I'll apply parts of speech(POS) to each word
# This is already in the dataset, but...
# I want NLTK to find the POS so this can be as NLTK specific as possible
# Maybe there's an easier way to do this - this takes about 20mins on Colab
for i in range((len(test_dataset))):
  pos_tags = pos_tag(test_dataset[i]['tokens'])
  named = ne_chunk(pos_tags) # Creates an NLTK tree
  lst = []

  # Traverse the tree
  for j in range(len(named)):

    # If there is a label
    if hasattr(named[j], 'label'):

      # Get length of chunk
      length = len(named[j])

      if named[j].label() == "PERSON":
        lst.append([named[j][0][0], 1])
        if length > 1:
          for k in range(1,length):
            lst.append([named[j][k][0], 2])
      elif named[j].label() == "ORGANIZATION":
        lst.append([named[j][0][0], 3])
        if length > 1:
          for k in range(1,length):
            lst.append([named[j][k][0], 4])
      elif named[j].label() == "LOCATION" or named[j].label() == "GSP":
        lst.append([named[j][0][0], 5])
        if length > 1:
          for k in range(1,length):
            lst.append([named[j][k][0], 6])
      else:
        lst.append([named[j][0][0], 7])
        if length > 1:
          for k in range(1,length):
            lst.append([named[j][k][0], 8])

    # No named entity
    else:
      lst.append([named[j][0], 0])

  # Add sentence list of lists to named entity list
  nltk_lst.append(lst)




4A) Now I'll evaluate the nltk's ne_chunk method's performance, inclusive of the B and I tags.

In [21]:
TP = 0
FP = 0
TN = 0
FN = 0
words = 0

# Loop through list, compare to actual tags
for i in range(len(nltk_lst)):
  labels = test_dataset[i]['ner_tags']
  nltk_sentence = nltk_lst[i]

  for j in range(len(nltk_sentence)):
    words += 1
    if nltk_sentence[j][1] == 0:
      if labels[j] == 0:
        TN += 1
      else:
        FN += 1
    else:
      if nltk_sentence[j][1] == labels[j]:
        TP += 1
      else:
        FP += 1



print(f"There are {words} words and {TP + FP + TN + FN} results")

precision = TP / (TP + FP)
recall = TP / (TP + FN)
accuracy = (TP + TN) / (TP + TN + FP + FN)
F1 = (2 * precision * recall) / (precision + recall)

print(f"Precision: {precision:.6f}")
print(f"Recall: {recall:.6f}")
print(f"Accuracy: {accuracy:.6f}")
print(f"F1: {F1:.6f}")

There are 46435 words and 46435 results
Precision: 0.450195
Recall: 0.626181
Accuracy: 0.878066
F1: 0.523802


4B) Next I'll evaluate the model independent of B and I tags

In [22]:
TP = 0
FP = 0
TN = 0
FN = 0
words = 0

# Loop through list, compare to actual tags
for i in range(len(nltk_lst)):
  labels = test_dataset[i]['ner_tags']
  nltk_sentence = nltk_lst[i]

  for j in range(len(nltk_sentence)):
    words += 1
    if nltk_sentence[j][1] == 0:
      if labels[j] == 0:
        TN += 1
      else:
        FN += 1
    else:
      if nltk_sentence[j][1] in [1,2] and labels[j] in [1,2]:
        TP += 1
      elif nltk_sentence[j][1] in [3,4] and labels[j] in [3,4]:
        TP += 1
      elif nltk_sentence[j][1] in [5,6] and labels[j] in [5,6]:
        TP += 1
      elif nltk_sentence[j][1] in [7,8] and labels[j] in [7,8]:
        TP += 1
      else:
        FP += 1



print(f"There are {words} words and {TP + FP + TN + FN} results")

precision = TP / (TP + FP)
recall = TP / (TP + FN)
accuracy = (TP + TN) / (TP + TN + FP + FN)
F1 = (2 * precision * recall) / (precision + recall)

print(f"Precision: {precision:.6f}")
print(f"Recall: {recall:.6f}")
print(f"Accuracy: {accuracy:.6f}")
print(f"F1: {F1:.6f}")

There are 46435 words and 46435 results
Precision: 0.473471
Recall: 0.637904
Accuracy: 0.881533
F1: 0.543523


5) My last method to expore was a transformer based method. I'll create a DistilBERT trained model. I used huggingface's resources at this link as a reference https://huggingface.co/docs/transformers/tasks/token_classification . I will first pre-process the data below.

In [23]:

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

label_all_tokens = True

# Although this data is already seperated into words, I will need to tokenize the data further
def tokenize_and_align_labels(data):

  # This further breaks down each word, if needed
  tokenized_inputs = tokenizer(data["tokens"], truncation=True, is_split_into_words=True)

  labels = []

  # This loops through each set and assigns the label accordingly
  for i, label in enumerate(data[f"ner_tags"]):
    word_ids = tokenized_inputs.word_ids(batch_index=i)
    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
      if word_idx is None:
        # This assigns -100 to tokens like CLS and SEP
        label_ids.append(-100)
      elif word_idx != previous_word_idx:
        # This labels the first token of a word
        label_ids.append(label[word_idx])
      else:
        # If more than one token in a word, set to -100
        label_ids.append(-100)
      previous_word_idx = word_idx

    labels.append(label_ids)

  tokenized_inputs["labels"] = labels
  return tokenized_inputs

# I will tokenize the entire dataset
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)




Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

5A) I will then create functions to evaluate this data.

In [24]:

seqeval = evaluate.load("seqeval")

#Create a list of all available labels
label_list = dataset["train"].features["ner_tags"].feature.names

# Function to compute metrics
def compute_metrics(p):
  predictions, labels = p
  predictions = np.argmax(predictions, axis=2)

  true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
  ]
  true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
  ]

  results = seqeval.compute(predictions=true_predictions, references=true_labels)
  return {
      "Precision": results["overall_precision"],
      "Recall": results["overall_recall"],
      "Accuracy": results["overall_accuracy"],
      "f1": results["overall_f1"],
  }

# Label mapping
id2label = {
  0: "0",
  1: "B-PER",
  2: "I-PER",
  3: "B-ORG",
  4: "I-ORG",
  5: "B-LOC",
  6: "I-LOC",
  7: "B-MISC",
  8: "I-MISC",
}

label2id = {
  "O": 0,
  "B-PER": 1,
  "I-PER": 2,
  "B-ORG": 3,
  "I-ORG": 4,
  "B-LOC": 5,
  "I-LOC": 6,
  "B-MISC": 7,
  "I-MISC": 8,
}

5B) Finally I will train on the training data and evaluate the model on the test data. My model may be found at https://huggingface.co/NickWoody/NLP_Model. ***PLEASE NOTE:*** If you're running this code you will likely want to change the `push_to_hub` arg to `False`

In [25]:
model = AutoModelForTokenClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=9, id2label=id2label, label2id=label2id)

training_args = TrainingArguments(
    output_dir="NLP_Model",
    learning_rate=2e-5,
    per_device_train_batch_size=20,
    per_device_eval_batch_size=20,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True, # Exclude this if you're not creating a model
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,Accuracy,F1
1,0.192766,0.097698,0.870564,0.885977,0.975450,0.878203
2,0.053171,0.109448,0.872937,0.898902,0.976871,0.885729
3,0.025196,0.110039,0.887631,0.902089,0.978658,0.894802


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'eval_loss': 0.09768934547901154,
 'eval_Precision': 0.8707151557334262,
 'eval_Recall': 0.8859773371104815,
 'eval_Accuracy': 0.9754495531387962,
 'eval_f1': 0.878279947345327,
 'eval_runtime': 2.4549,
 'eval_samples_per_second': 1406.555,
 'eval_steps_per_second': 70.47,
 'epoch': 3.0}